# Adaptive Guardrail Layer (AGL) - Data Fetching Notebook

In [ ]:
import os
import json
from pathlib import Path
import pandas as pd
from dotenv import load_dotenv
from huggingface_hub import login
from datasets import load_dataset

# Create a HF key: https://huggingface.co/docs/hub/en/security-tokens
# add env variable: echo "HF_TOKEN=<your-key>" > .env
env_path = Path("../.env")
load_dotenv(env_path)

# auth to hugginface API
hf_token = os.getenv("HF_TOKEN")
login(hf_token)

INPUT_PATH = "../../data/raw/"
OUTPT_PATH = "../../data/processed/"

### Dataset Structure

In [ ]:
# all datasets used in the project
DATASETS = [
    {
        "name": "WildGuardMix",
        "loader": lambda: pd.concat([
            pd.read_parquet("hf://datasets/allenai/wildguardmix/train/wildguard_train.parquet"),
            pd.read_parquet("hf://datasets/allenai/wildguardmix/test/wildguard_test.parquet"),
        ], ignore_index=True),
        "prompt_col": "prompt",
        "label_col": "prompt_harm_label",
        "malicious_value": "harmful",
        "benign_value": "unharmful",
    },
    {
        "name": "Malicious LLM Prompts",
        "loader": lambda: pd.concat([
            pd.read_parquet("hf://datasets/codesagar/malicious-llm-prompts/data/train-00000-of-00001-2279f74035821199.parquet"),
            pd.read_parquet("hf://datasets/codesagar/malicious-llm-prompts/data/validation-00000-of-00001-a3f9d7ca008d6126.parquet"),
            pd.read_parquet("hf://datasets/codesagar/malicious-llm-prompts/data/test-00000-of-00001-fec3804704adbfa8.parquet"),
        ], ignore_index=True),
        "prompt_col": "prompt",
        "label_col": "malicious",
        "malicious_value": True,
        "benign_value": False,
    },
    {
        "name": "MPDD",
        "loader": lambda: pd.read_csv(Path(INPUT_PATH) / "MPDD.csv"),
        "prompt_col": "Prompt",
        "label_col": "isMalicious",
        "malicious_value": 1,
        "benign_value": 0,
    },
    {
        "name": "Prompt Injection & Benign Prompt Dataset",
        "loader": lambda: pd.read_json(Path(INPUT_PATH) / "Prompt_INJECTION_And_Benign_DATASET.jsonl", lines=True),
        "prompt_col": "prompt",
        "label_col": "label",
        "malicious_value": "malicious",
        "benign_value": "benign",
    },
    {
        "name": "malicious-llm-prompts-v4",
        "loader": lambda: pd.concat([
            pd.read_parquet("hf://datasets/codesagar/malicious-llm-prompts-v4/data/train-00000-of-00001-a56d4b725b007225.parquet"),
            pd.read_parquet("hf://datasets/codesagar/malicious-llm-prompts-v4/data/test-00000-of-00001-648610b8af893e5a.parquet"),
        ], ignore_index=True),
        "prompt_col": "prompt",
        "label_col": "malicious",
        "malicious_value": True,
        "benign_value": False,
    },
    {
        "name": "deepset/prompt-injections",
        "loader": lambda: pd.concat([
            pd.read_parquet("hf://datasets/deepset/prompt-injections/data/train-00000-of-00001-9564e8b05b4757ab.parquet"),
            pd.read_parquet("hf://datasets/deepset/prompt-injections/data/test-00000-of-00001-701d16158af87368.parquet"),
        ], ignore_index=True),
        "prompt_col": "text",
        "label_col": "label",
        "malicious_value": 1,
        "benign_value": 0,
    },
    {
        "name": "jackhhao/jailbreak-classification",
        "loader": lambda: pd.concat([
            pd.read_csv("hf://datasets/jackhhao/jailbreak-classification/balanced/jailbreak_dataset_train_balanced.csv"),
            pd.read_csv("hf://datasets/jackhhao/jailbreak-classification/balanced/jailbreak_dataset_test_balanced.csv"),
        ], ignore_index=True),
        "prompt_col": "prompt",
        "label_col": "type",
        "malicious_value": "jailbreak",
        "benign_value": "benign",
    },
]

### Dataset Normalization and Merge Process

In [ ]:
merged_parts = []
summary_rows = []

for ds in DATASETS:
    df = ds["loader"]().copy()

    # keep only rows with the two expected label values
    df = df[df[ds["label_col"]].isin([ds["malicious_value"], ds["benign_value"]])].copy()

    # normalize schema (malicious: 1, benign: 0)
    df["prompt"] = df[ds["prompt_col"]].astype(str).str.strip()
    df["label"] = df[ds["label_col"]].map({
        ds["malicious_value"]: 1,
        ds["benign_value"]: 0
    })
    df["source_dataset"] = ds["name"] # which dataset each row came from

    # keep only the standardized columns
    merged_parts.append(df[["prompt", "label", "source_dataset"]])

    # collect summary information about the processed dataset
    summary_rows.append({
        "dataset": ds["name"],
        "rows_kept": len(df),
        "malicious_count": int((df["label"] == 1).sum()),
        "benign_count": int((df["label"] == 0).sum()),
        "prompt_column_used": ds["prompt_col"],
        "label_column_used": ds["label_col"],
    })

### Per-dataset summary

In [ ]:
summary_df = pd.DataFrame(summary_rows)
display(summary_df)

### Merged dataset shape

In [ ]:
merged_df = pd.concat(merged_parts, ignore_index=True)
print(merged_df.shape)

### Overall label distribution

In [ ]:
print(merged_df["label"].value_counts().rename(index={1: "malicious", 0: "benign"}))

### Sample rows

In [ ]:
display(merged_df.head())

### Save Merged dataset

In [ ]:
output_file = Path(OUTPT_PATH) / "dataset_merged.csv"
merged_df.to_csv(output_file, index=False)

# verify file creation
if output_file.exists():
    print(f"File created successfully: {output_file}")
else:
    raise FileNotFoundError(f"File was NOT created: {output_file}")